In [ ]:
#Instalaciones
# %pip install optuna
#%pip install "accelerate>=1.1.0"

# Importaciones
import gc
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# CONFIGURACIÓN DE REPRODUCIBILIDAD

In [ ]:
# =====================================================
#  CONFIGURACIÓN DE REPRODUCIBILIDAD
# =====================================================

import random
import numpy as np
import torch
from transformers import set_seed

# Fijar una semilla común para todas las librerías
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("Semilla fijada:", SEED)

# CARGA DE LOS CONJUNTOS DE DATOS

In [ ]:
# =====================================================
#  CARGA DE LOS CONJUNTOS DE DATOS
# =====================================================

# Carpeta donde están guardados los splits
carpeta_splits = r"./Bases de Datos Splits/"

# Cargar Train balanceados
train_DATD_bal = pd.read_csv(carpeta_splits + "train_DATD_balanceado.csv")
train_SDCNL_bal = pd.read_csv(carpeta_splits + "train_SDCNL_balanceado.csv")
train_DU_bal = pd.read_csv(carpeta_splits + "train_DU_balanceado.csv")

# Cargar Validation y Test originales
val_DATD = pd.read_csv(carpeta_splits + "val_DATD.csv")
test_DATD = pd.read_csv(carpeta_splits + "test_DATD.csv")

val_SDCNL = pd.read_csv(carpeta_splits + "val_SDCNL.csv")
test_SDCNL = pd.read_csv(carpeta_splits + "test_SDCNL.csv")

val_DU = pd.read_csv(carpeta_splits + "val_DU.csv")
test_DU = pd.read_csv(carpeta_splits + "test_DU.csv")

print("Datasets cargados correctamente.")

# Tokenización, Padding y Truncation

In [ ]:
# =====================================================
#  SELECCIÓN DEL MODELO TRANSFORMER
# =====================================================

# Modelo 1: RoBERTa Base
# nombre_modelo = "roberta-base"

# Modelo 2: DeBERTa v3 Base
nombre_modelo = "microsoft/deberta-v3-base"

# Crear un nombre corto para carpetas y archivos
if nombre_modelo == "roberta-base":
    nombre_corto = "RoBERTa"
elif nombre_modelo == "microsoft/deberta-v3-base":
    nombre_corto = "DeBERTa"
else:
    raise ValueError("Transformer no reconocido.")

# Cargar el tokenizador correspondiente
tokenizer = AutoTokenizer.from_pretrained(nombre_modelo)

# Longitud máxima
MAX_LENGTH = 128

print("Modelo seleccionado:", nombre_modelo)
print("Nombre corto:", nombre_corto)

In [ ]:
# =====================================================
# RUTA DE GUARDADO DE RESULTADOS
# =====================================================

carpeta_resultados = r"./Resultados Optuna/"

print("Carpeta de resultados:", carpeta_resultados)

In [ ]:
# =====================================================
#  FUNCIÓN DE TOKENIZACIÓN, PADDING Y TRUNCATION
# =====================================================

def tokenizar(df):
    return tokenizer(
        df["Text"].tolist(), # Le estamos pasando al tokenizer únicamente la columna del Texto
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
# =====================================================
#  TOKENIZACIÓN DE TRAIN, VALIDATION Y TEST
# =====================================================

# DATD
tokens_train_DATD = tokenizar(train_DATD_bal)
tokens_val_DATD = tokenizar(val_DATD)
tokens_test_DATD = tokenizar(test_DATD)

# SDCNL
tokens_train_SDCNL = tokenizar(train_SDCNL_bal)
tokens_val_SDCNL = tokenizar(val_SDCNL)
tokens_test_SDCNL = tokenizar(test_SDCNL)

# DU
tokens_train_DU = tokenizar(train_DU_bal)
tokens_val_DU = tokenizar(val_DU)
tokens_test_DU = tokenizar(test_DU)

print("Tokenización completada correctamente.")

In [ ]:
# =====================================================
# COMPROBAR UNA INSTANCIA ANTES Y DESPUÉS DE TOKENIZAR
# =====================================================

print("========== ANTES DE TOKENIZAR ==========")
print("Texto:")
print(train_DATD_bal["Text"].iloc[0])

print("\nLabel:")
print(train_DATD_bal["Label"].iloc[0])

print("\n========== DESPUÉS DE TOKENIZAR ==========")
print("Input IDs:")
print(tokens_train_DATD["input_ids"][0])

print("\nAttention Mask:")
print(tokens_train_DATD["attention_mask"][0])

print("\nLabel:")
print(train_DATD_bal["Label"].iloc[0])

In [ ]:
# =====================================================
# 6. COMPROBACIÓN DE TOKENIZACIÓN, PADDING Y TRUNCATION
# =====================================================

print("Modelo utilizado:", nombre_modelo)
print("Longitud máxima:", MAX_LENGTH)

print("\nDATD Train:")
print("Número de instancias:", len(tokens_train_DATD["input_ids"]))
print("Longitud de una secuencia:", len(tokens_train_DATD["input_ids"][0]))

print("\nSDCNL Train:")
print("Número de instancias:", len(tokens_train_SDCNL["input_ids"]))
print("Longitud de una secuencia:", len(tokens_train_SDCNL["input_ids"][0]))

print("\nDU Train:")
print("Número de instancias:", len(tokens_train_DU["input_ids"]))
print("Longitud de una secuencia:", len(tokens_train_DU["input_ids"][0]))

# Mostrar un ejemplo de la tokenización
print("\nPrimeros 20 input_ids de DATD:")
print(tokens_train_DATD["input_ids"][0][:20])

print("\nPrimeros 20 valores de attention_mask:")
print(tokens_train_DATD["attention_mask"][0][:20])

# PREPARACIÓN DE LOS DATOS PARA EL TRAINER

In [ ]:
# =====================================================
# PREPARACIÓN DE LOS DATOS PARA EL TRAINER
# =====================================================

# Dataset permite crear conjuntos de datos que PyTorch y
# el Trainer de Hugging Face pueden utilizar durante el entrenamiento
from torch.utils.data import Dataset


# Crear una clase para juntar los textos tokenizados con sus etiquetas
class DatasetTexto(Dataset):

    def __init__(self, tokens, labels):
        # Guardar los input_ids y attention_mask obtenidos en la tokenización
        self.tokens = tokens

        # Guardar las etiquetas correspondientes a cada texto
        self.labels = labels.tolist()

    def __len__(self):
        # Devolver el número total de instancias del dataset
        return len(self.labels)

    def __getitem__(self, idx):
        # Devolver una instancia concreta con:
        # input_ids + attention_mask + etiqueta correcta
        return {
            "input_ids": torch.tensor(self.tokens["input_ids"][idx]),
            "attention_mask": torch.tensor(self.tokens["attention_mask"][idx]),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# =====================================================
# CREACIÓN DE LOS DATASETS PARA EL TRAINER
# =====================================================

# Juntar los tokens de cada conjunto con sus etiquetas correspondientes.
# Train utiliza los conjuntos balanceados.
# Validation y Test utilizan los conjuntos originales sin balancear.

# DATD
dataset_train_DATD = DatasetTexto(tokens_train_DATD, train_DATD_bal["Label"])
dataset_val_DATD = DatasetTexto(tokens_val_DATD, val_DATD["Label"])
dataset_test_DATD = DatasetTexto(tokens_test_DATD, test_DATD["Label"])

# SDCNL
# Las etiquetas originales son:
# 1 = depresión
# 2 = suicidio
#
# Como el modelo es binario, Hugging Face necesita etiquetas 0 y 1.
# Por eso se hace este cambio únicamente para el entrenamiento:
# 1 -> 0 (depresión)
# 2 -> 1 (suicidio)

labels_train_SDCNL = train_SDCNL_bal["Label"].map({1: 0, 2: 1})
labels_val_SDCNL = val_SDCNL["Label"].map({1: 0, 2: 1})
labels_test_SDCNL = test_SDCNL["Label"].map({1: 0, 2: 1})

dataset_train_SDCNL = DatasetTexto(tokens_train_SDCNL, labels_train_SDCNL)
dataset_val_SDCNL = DatasetTexto(tokens_val_SDCNL, labels_val_SDCNL)
dataset_test_SDCNL = DatasetTexto(tokens_test_SDCNL, labels_test_SDCNL)

# DU
dataset_train_DU = DatasetTexto(tokens_train_DU, train_DU_bal["Label"])
dataset_val_DU = DatasetTexto(tokens_val_DU, val_DU["Label"])
dataset_test_DU = DatasetTexto(tokens_test_DU, test_DU["Label"])

print("Datasets preparados correctamente para el Trainer.")

In [ ]:
# =====================================================
# COMPROBACIÓN DE LOS DATOS PREPARADOS
# =====================================================

# Seleccionar la primera instancia del Train de DATD
ejemplo = dataset_train_DATD[0]

# Mostrar los tres elementos que recibirá el modelo
print("========== EJEMPLO DATD ==========")

print("\nInput IDs:")
print(ejemplo["input_ids"])

print("\nAttention Mask:")
print(ejemplo["attention_mask"])

print("\nLabel:")
print(ejemplo["labels"])

# Comprobar también el número de instancias de cada Train
print("\n========== TAMAÑO DE LOS DATASETS ==========")
print("DATD Train:", len(dataset_train_DATD))
print("SDCNL Train:", len(dataset_train_SDCNL))
print("DU Train:", len(dataset_train_DU))

# DEFINICIÓN DE LAS MÉTRICAS

In [ ]:
# =====================================================
#  DEFINICIÓN DE LA MÉTRICA F1
# =====================================================

import numpy as np
from sklearn.metrics import f1_score

# =====================================================
# F1 PARA LOS MODELOS BINARIOS: DATD Y SDCNL
# =====================================================

def compute_metrics_binario(eval_pred):
    # Obtener las puntuaciones del modelo y las etiquetas reales
    logits, labels = eval_pred

    # Convertir las puntuaciones en clases predichas
    predictions = np.argmax(logits, axis=-1)

    # Calcular el F1 de la clase positiva (clase 1)
    f1 = f1_score(labels, predictions, average="binary", pos_label=1)

    return {"f1": f1}

# =====================================================
# F1 PARA EL MODELO MULTICLASE: DU
# =====================================================

def compute_metrics_multiclase(eval_pred):
    # Obtener las puntuaciones del modelo y las etiquetas reales
    logits, labels = eval_pred

    # Convertir las puntuaciones en clases predichas
    predictions = np.argmax(logits, axis=-1)

    # Calcular el F1 de cada clase y hacer la media
    f1 = f1_score(labels, predictions, average="macro")

    return {"f1": f1}

In [ ]:
# =====================================================
#  CONFIGURACIÓN DEL TRAINER Y EARLY STOPPING
# =====================================================

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback


# BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - DATD

In [ ]:
# =====================================================
# 14. BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - DATD
# =====================================================

import optuna

print("Versión de Optuna:", optuna.__version__)

In [ ]:
# =====================================================
# FUNCIÓN OBJETIVO DE OPTUNA - DATD
# =====================================================

def objective_DATD(trial):

    # Optuna selecciona una combinación de hiperparámetros
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    learning_rate = trial.suggest_categorical("learning_rate", [1e-5, 3e-5, 5e-5])
    weight_decay = trial.suggest_float("weight_decay", 0.01, 0.1)
    epochs = trial.suggest_int("epochs", 10, 15)

    print("\n========================================")
    print(f"TRIAL {trial.number}")
    print("========================================")
    print("Batch size:", batch_size)
    print("Learning rate:", learning_rate)
    print("Weight decay:", weight_decay)
    print("Épocas máximas:", epochs)

    # Reiniciar la semilla antes de crear el modelo para que cada trial parta de una inicialización reproducible
    set_seed(SEED)
    
    # Crear un modelo nuevo para cada trial
    model_trial = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=2,
        dtype=torch.float32
    )

    # Configurar el entrenamiento de este trial
    args_trial = TrainingArguments(
        output_dir=f"./resultados/{nombre_corto}/DATD_trial_{trial.number}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        weight_decay=weight_decay,
        num_train_epochs=epochs,
        
        warmup_steps=100,
        max_grad_norm=1.0,
        
        seed=SEED,
        save_total_limit=1,
        logging_strategy="epoch",
        report_to="none"
    )

    # Crear el Trainer de este trial
    trainer_trial = Trainer(
        model=model_trial,
        args=args_trial,
        train_dataset=dataset_train_DATD,
        eval_dataset=dataset_val_DATD,
        compute_metrics=compute_metrics_binario,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # Entrenar
    trainer_trial.train()

    # Devolver a Optuna el MEJOR F1 alcanzado durante el trial
    mejor_f1 = trainer_trial.state.best_metric

    print(f"\nMejor F1 del trial {trial.number}: {mejor_f1}")

    # Liberar memoria antes de comenzar el siguiente trial
    del trainer_trial
    del model_trial
    gc.collect()
    torch.cuda.empty_cache()

    return mejor_f1

In [ ]:
# =====================================================
# EJECUTAR OPTUNA - DATD
# =====================================================

# Fijar la semilla de Optuna para hacer reproducible
# la búsqueda de hiperparámetros
sampler = optuna.samplers.TPESampler(seed=SEED)

# Crear el estudio de Optuna
study_DATD = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# Ejecutar la búsqueda de hiperparámetros
study_DATD.optimize(
    objective_DATD,
    n_trials=20
)

In [ ]:
# =====================================================
# MEJORES HIPERPARÁMETROS ENCONTRADOS - DATD
# =====================================================

print("Mejor F1:", study_DATD.best_value)

print("\nMejores hiperparámetros:")
print(study_DATD.best_params)

In [ ]:
# =====================================================
# GUARDAR RESULTADOS DE OPTUNA - DATD
# =====================================================

resultados_optuna_DATD = study_DATD.trials_dataframe()

resultados_optuna_DATD.to_csv(
    carpeta_resultados + f"resultados_optuna_{nombre_corto}_DATD.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    f"Resultados de Optuna de {nombre_corto} - DATD "
    "guardados correctamente."
)

# BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - SDCNL

In [ ]:
# =====================================================
#  BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - SDCNL
# =====================================================

In [ ]:
# =====================================================
# FUNCIÓN OBJETIVO DE OPTUNA - SDCNL
# =====================================================

def objective_SDCNL(trial):

    # Optuna selecciona una combinación de hiperparámetros
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    learning_rate = trial.suggest_categorical("learning_rate", [1e-5, 3e-5, 5e-5])
    weight_decay = trial.suggest_float("weight_decay", 0.01, 0.1)
    epochs = trial.suggest_int("epochs", 10, 15)

    print("\n========================================")
    print(f"TRIAL {trial.number} - SDCNL")
    print("========================================")
    print("Batch size:", batch_size)
    print("Learning rate:", learning_rate)
    print("Weight decay:", weight_decay)
    print("Épocas máximas:", epochs)

    # Reiniciar la semilla antes de crear el modelo para que cada trial parta de una inicialización reproducible
    set_seed(SEED)

    # Crear un modelo nuevo para cada trial
    # SDCNL es una clasificación binaria:
    # 0 = depresión
    # 1 = suicidio
    model_trial = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=2,
        dtype=torch.float32
    )

    # Configuración del entrenamiento para este trial
    args_trial = TrainingArguments(
        output_dir=f"./resultados/{nombre_corto}/SDCNL_trial_{trial.number}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        weight_decay=weight_decay,
        num_train_epochs=epochs,

        warmup_steps=100,
        max_grad_norm=1.0,

        seed=SEED,
        save_total_limit=1,
        logging_strategy="epoch",
        report_to="none"
    )

    # Crear el Trainer para este trial
    trainer_trial = Trainer(
        model=model_trial,
        args=args_trial,
        train_dataset=dataset_train_SDCNL,
        eval_dataset=dataset_val_SDCNL,
        compute_metrics=compute_metrics_binario,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # Entrenar el modelo
    trainer_trial.train()

    # Obtener el mejor F1 alcanzado durante el trial
    mejor_f1 = trainer_trial.state.best_metric

    print(f"\nMejor F1 del trial {trial.number}: {mejor_f1}")


    # Liberar memoria antes de comenzar el siguiente trial
    del trainer_trial
    del model_trial
    gc.collect()
    torch.cuda.empty_cache()
    
    return mejor_f1

In [ ]:
# =====================================================
# EJECUTAR OPTUNA - SDCNL
# =====================================================

# Fijar la semilla del algoritmo de búsqueda de Optuna
sampler_SDCNL = optuna.samplers.TPESampler(seed=SEED)

# Crear el estudio
study_SDCNL = optuna.create_study(
    direction="maximize",
    sampler=sampler_SDCNL
)

# Ejecutar 20 trials
study_SDCNL.optimize(
    objective_SDCNL,
    n_trials=20
)

In [ ]:
# =====================================================
# MEJORES HIPERPARÁMETROS ENCONTRADOS - SDCNL
# =====================================================

print("Mejor F1:", study_SDCNL.best_value)

print("\nMejores hiperparámetros:")
print(study_SDCNL.best_params)

In [ ]:
# =====================================================
# GUARDAR RESULTADOS DE OPTUNA - SDCNL
# =====================================================

resultados_optuna_SDCNL = study_SDCNL.trials_dataframe()

resultados_optuna_SDCNL.to_csv(
    carpeta_resultados + f"resultados_optuna_{nombre_corto}_SDCNL.csv",
    index=False,
    encoding="utf-8-sig"
)

print(resultados_optuna_SDCNL)
print(
    f"\nResultados de Optuna de {nombre_corto} - SDCNL "
    "guardados correctamente."
)

# BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - DU

In [ ]:
# =====================================================
#  BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - DU
# =====================================================

In [ ]:
# =====================================================
# FUNCIÓN OBJETIVO DE OPTUNA - DU
# =====================================================

def objective_DU(trial):

    # Optuna selecciona una combinación de hiperparámetros
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    learning_rate = trial.suggest_categorical("learning_rate", [1e-5, 3e-5, 5e-5])
    weight_decay = trial.suggest_float("weight_decay", 0.01, 0.1)
    epochs = trial.suggest_int("epochs", 10, 15)

    print("\n========================================")
    print(f"TRIAL {trial.number} - DU")
    print("========================================")
    print("Batch size:", batch_size)
    print("Learning rate:", learning_rate)
    print("Weight decay:", weight_decay)
    print("Épocas máximas:", epochs)

    # Reiniciar la semilla antes de crear el modelo para que cada trial parta de una inicialización reproducible
    set_seed(SEED)
    
    # Crear un modelo nuevo para cada trial
    # DU es una clasificación multiclase:
    # 0 = sano
    # 1 = depresión
    # 2 = suicidio
    
    model_trial = AutoModelForSequenceClassification.from_pretrained(
        nombre_modelo,
        num_labels=3,
        dtype=torch.float32
    )

    # Configuración del entrenamiento para este trial
    args_trial = TrainingArguments(
        output_dir=f"./resultados/{nombre_corto}/DU_trial_{trial.number}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        weight_decay=weight_decay,
        num_train_epochs=epochs,

        warmup_steps=100,
        max_grad_norm=1.0,
        
        seed=SEED,
        save_total_limit=1,
        logging_strategy="epoch",
        report_to="none"
    )

    # Crear el Trainer para este trial
    trainer_trial = Trainer(
        model=model_trial,
        args=args_trial,
        train_dataset=dataset_train_DU,
        eval_dataset=dataset_val_DU,
        compute_metrics=compute_metrics_multiclase,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # Entrenar el modelo
    trainer_trial.train()

    # Obtener el mejor Macro-F1 alcanzado durante el trial
    mejor_f1 = trainer_trial.state.best_metric

    print(f"\nMejor Macro-F1 del trial {trial.number}: {mejor_f1}")

    # Liberar memoria antes de comenzar el siguiente trial
    del trainer_trial
    del model_trial
    gc.collect()
    torch.cuda.empty_cache()
    
    return mejor_f1

In [ ]:
# =====================================================
# EJECUTAR OPTUNA - DU
# =====================================================

# Fijar la semilla de Optuna para hacer reproducible la búsqueda
sampler_DU = optuna.samplers.TPESampler(seed=SEED)

# Crear el estudio
study_DU = optuna.create_study(
    direction="maximize",
    sampler=sampler_DU
)

# Ejecutar 20 trials
study_DU.optimize(
    objective_DU,
    n_trials=20
)

In [ ]:
# =====================================================
# MEJORES HIPERPARÁMETROS ENCONTRADOS - DU
# =====================================================

print("Mejor Macro-F1:", study_DU.best_value)

print("\nMejores hiperparámetros:")
print(study_DU.best_params)

In [ ]:
# =====================================================
# GUARDAR RESULTADOS DE OPTUNA - DU
# =====================================================

resultados_optuna_DU = study_DU.trials_dataframe()

resultados_optuna_DU.to_csv(
    carpeta_resultados + f"resultados_optuna_{nombre_corto}_DU.csv",
    index=False,
    encoding="utf-8-sig"
)

print(resultados_optuna_DU)
print(
    f"\nResultados de Optuna de {nombre_corto} - DU "
    "guardados correctamente."
)